# Building a Quant Trading Strategy - Part 2

In [ ]:
# signal = model(x)
# orders = strategy(signal)
# execute(orders)

### Strategy Types

In [ ]:
# 1 Maker strategies => providing liquidity => placing limit orders and aiming to be compensated for it
# 2 Taker strategies => taking liquidity => market orders that consume liquidity => what we are going to focus on

In [ ]:
# Key Questions for our taker strategy

# 1. Entry/Exit
# 2. Trade Sizing
# 3. Leverage

In [ ]:
# The key goal is to create a strategy that maximises profits from the model's statistical edge.

### Import Libraries

In [ ]:
# Data and analysis libraries
import polars as pl                         # Fast dataframes for financial data
import numpy as np                          # Numerical computing library
from datetime import datetime, timedelta    # Date and time operations
import random


# Machine learning libraries
import torch                                # PyTorch framework
import torch.nn as nn                       # Neural network modules
import torch.optim as optim                 # Optimization algorithms
import research                             # Model building and training utilities
import models                               # Model architecture definitions


# Visualization
import altair as alt                        # Interactive visualization library

# Data source for QQQ (stock ETF)
import yfinance as yf                       # Yahoo Finance market data

### Load Model

In [ ]:
# The model predicts the next day's (1-day ahead) log return for QQQ.
# It uses the 3 most recent daily log return lags as features.
# We saved this model in Part 1 code — now we load it here to build our strategy.

In [ ]:
model = models.LinearModel(3)
# security alert
model.load_state_dict(torch.load('qqq_model_weights.pth', weights_only=True))
model.eval()

### Model Parameters

In [ ]:
research.print_model_params(model)

### What is Mean Reversion?

In [ ]:
# QQQ daily log returns exhibit mild mean reversion — after a big move up, the next day tends to be slightly negative, and vice versa.
# This is the statistical edge our model is trying to capture.
series = [-0.0089, 0.0073, -0.0082, 0.0091, -0.0076, 0.0068]
mu = np.mean(series)
mean_reversion_df = pl.DataFrame({'log_return': series, 'mean': mu})
mean_reversion_df

In [ ]:
research.plot_multiple_lines(mean_reversion_df, ['log_return','mean'], 'Mean Reversion - QQQ Daily Log Returns')

### Interpretability - Linear Model

In [ ]:
# A linear model is y = w1*x1 + w2*x2 + w3*x3 + b
# w1, w2, w3 are the weights for each lag feature; b is the bias
# Negative w1 => mean reversion (if yesterday was +1%, model predicts next day to be slightly negative)

def linear_model(x1, x2, x3):
    # Example weights from trained model (actual values come from model.linear.weight)
    w1, w2, w3, b = -0.12845, 0.03124, -0.01873, 0.00047
    return w1 * x1 + w2 * x2 + w3 * x3 + b

# If yesterday QQQ dropped 0.89%, what does the model predict for today?
linear_model(-0.0089, 0.0073, -0.0082)

In [ ]:
# If yesterday QQQ rose 0.91%?
linear_model(0.0091, -0.0082, 0.0073)

In [ ]:
# If yesterday was essentially flat?
linear_model(0.000002, 0.0, 0.0)

In [ ]:
# At zero — the model just returns the bias term
linear_model(0.0, 0.0, 0.0)

### Interpretability - Non Linear Model

In [ ]:
def nn_model(x1, x2, x3):
    x = torch.tensor([x1, x2, x3])
    W = torch.tensor([
        [0.0804, -0.0148, -0.0352],
        [0.0678,  0.0379,  0.0014],
        [-0.1330,  0.8045,  0.0512],
    ])
    b = torch.tensor([0.1642, -0.0832, 0.0421])
    hidden = torch.tanh(W @ x + b)
    w_out = torch.tensor([0.4521, -0.3107, 0.2984])
    return (w_out @ hidden).item()

nn_model(-0.0089, 0.0073, -0.0082)

In [ ]:
# no interpretability
nn_model(0.0091, -0.0082, 0.0073)

In [ ]:
nn_model(0.0, 0.0, 0.0)

## Strategy Development!

### Load Time Series

In [ ]:
# yf.download(sym, start=start_date, end=end_date, interval=time_interval)

In [ ]:
sym = 'QQQ'
time_interval = '1d'

start_date = datetime(2023, 10, 29)
end_date = datetime(2025, 10, 9)

raw_df = yf.download(sym, start=start_date, end=end_date, interval=time_interval)

ts = pl.DataFrame({
    'datetime': raw_df.index.to_pydatetime().tolist(),
    'open':     raw_df['Open'].values.flatten().astype(float),
    'high':     raw_df['High'].values.flatten().astype(float),
    'low':      raw_df['Low'].values.flatten().astype(float),
    'close':    raw_df['Close'].values.flatten().astype(float),
}).with_columns(pl.col('datetime').cast(pl.Datetime)).sort('datetime')
ts

### Add Target and Features

In [ ]:
forecast_horizon = 1
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=3)
ts

### Time Split

In [ ]:
test_size = 0.25
_, trades = research.timeseries_split(ts, test_size)
trades

## Strategy Decision # 1: Entry / Exit Signal

In [ ]:
# q1: when do we get in? entry signal
# q2: when do we get out? exit signal

In [ ]:
# 1. Time Based
# 2. Predicate Based

In [ ]:
# predicate example => we only want to trade if our y_hat is above or below a certain threshold

In [ ]:
# time based => each row represents a roundtrip trade.
# trade to open position at start of day (open price), close the position at end of day (close price).
# each row = 2 trades.

In [ ]:
trades

### Add Model's Predictions

In [ ]:
target = 'close_log_return'
features = [f'{target}_lag_1', f'{target}_lag_2', f'{target}_lag_3']
trades = research.add_model_predictions(trades, model, features)
trades

### Add Directional Signal

In [ ]:
trades = trades.with_columns(pl.col('y_hat').sign().alias('dir_signal'))
trades

### Calculate Trade Log Return

In [ ]:
trades = trades.with_columns((pl.col('close_log_return') * pl.col('dir_signal')).alias('trade_log_return'))
trades

### Calculate Cumulative Trade Log Return

In [ ]:
trades = trades.with_columns(pl.col('trade_log_return').cum_sum().alias('cum_trade_log_return'))
trades

### Display Equity Curve (Log Space)

In [ ]:
research.plot_column(trades, 'cum_trade_log_return')

## Key Strategy Decision #2: Trade Sizing

In [ ]:
# 1. Constant Trade Size
# 2. Compounding Trade Size

In [ ]:
capital = 100
ratio = 1.0  # you could experiment with different ratios like Kelly Criterion
trade_value = ratio * capital

# entry trade value and exit trade value
trades = trades.with_columns(
    pl.lit(trade_value).alias('entry_trade_value'),
    (trade_value * pl.col('trade_log_return').exp()).alias('exit_trade_value'),
    (trade_value / pl.col('open')).alias('trade_qty'),
).with_columns(
    (pl.col('trade_qty') * pl.col('dir_signal')).alias('signed_trade_qty'),
)

trades.select('datetime', 'open', 'close', 'trade_log_return', 'y_hat', 'entry_trade_value', 'exit_trade_value', 'signed_trade_qty')

### Add Trade Gross PnL

In [ ]:
trades = trades.with_columns((pl.col('exit_trade_value') - pl.col('entry_trade_value')).alias('trade_gross_pnl'))
trades.select('datetime', 'open', 'close', 'trade_log_return', 'y_hat', 'entry_trade_value', 'exit_trade_value', 'signed_trade_qty')

### Add Transaction Fees

In [ ]:
# For US equity ETFs (like QQQ), many brokers offer commission-free trading.
# However, there are still regulatory fees and spread costs.
maker_fee = 0.0000  # commission-free limit orders
taker_fee = 0.0001  # ~1 basis point for market orders (regulatory fees / spread)

trades = trades.with_columns(
    (pl.col('entry_trade_value') * taker_fee + pl.col('exit_trade_value') * taker_fee).alias('taker_fee'),
    (pl.col('entry_trade_value') * maker_fee + pl.col('exit_trade_value') * maker_fee).alias('maker_fee')
)

trades.select('datetime', 'open', 'close', 'trade_log_return', 'y_hat', 'entry_trade_value', 'exit_trade_value', 'signed_trade_qty', 'trade_gross_pnl', 'maker_fee', 'taker_fee')

### Calculate Trade Net PnL

In [ ]:
trades = trades.with_columns(
    (pl.col('trade_gross_pnl') - pl.col('taker_fee')).alias('trade_net_taker_pnl'),
    (pl.col('trade_gross_pnl') - pl.col('maker_fee')).alias('trade_net_maker_pnl'),
)

trades.select('datetime', 'open', 'close', 'trade_log_return', 'y_hat', 'entry_trade_value', 'exit_trade_value', 'signed_trade_qty', 'trade_gross_pnl', 'trade_net_taker_pnl', 'trade_net_maker_pnl')

### Display Equity Curves for Constant Sizing

In [ ]:
def equity_curve(capital, col_name, suffix):
    return (capital + (pl.col(col_name).cum_sum())).alias(f'equity_curve_{suffix}')

trades = trades.with_columns(
    equity_curve(capital, 'trade_net_taker_pnl', 'taker'),
    equity_curve(capital, 'trade_net_maker_pnl', 'maker'),
    equity_curve(capital, 'trade_gross_pnl', 'gross'),
)
trades.select('datetime', 'equity_curve_gross', 'equity_curve_taker', 'equity_curve_maker')

In [ ]:
research.plot_static_timeseries(trades, sym, 'equity_curve_taker', time_interval)

In [ ]:
research.plot_static_timeseries(trades, sym, 'equity_curve_maker', time_interval)

### Calculate Total Net Return using Constant Sizing

In [ ]:
constant_sizing_net_return = trades['equity_curve_taker'][-1] / capital - 1
constant_sizing_net_return

## Experiment with Compounding Trade Sizes

In [ ]:
log_return_1 = 0.003201
pnl1 = capital * np.exp(log_return_1)
pnl1

In [ ]:
log_return_2 = 0.004518
pnl2 = pnl1 * np.exp(log_return_2)
pnl2

In [ ]:
log_return_3 = 0.001124
pnl3 = pnl2 * np.exp(log_return_3)
pnl3

In [ ]:
((capital * np.exp(0.003201)) * np.exp(0.004518)) * np.exp(0.001124)

### Time Additivity

In [ ]:
capital * np.exp(log_return_1 + log_return_2 + log_return_3)

### Add Compounding Trade Sizes

In [ ]:
trades = trades.with_columns(
    ((pl.col('cum_trade_log_return').exp()) * capital).shift().fill_null(capital).alias('entry_trade_value'),
    ((pl.col('cum_trade_log_return').exp()) * capital).alias('exit_trade_value'),
).with_columns(
    (pl.col('entry_trade_value') / pl.col('open') * pl.col('dir_signal')).alias('signed_trade_qty'),
    (pl.col('exit_trade_value') - pl.col('entry_trade_value')).alias('trade_gross_pnl'),
)
trades.select('datetime', 'open', 'close', 'trade_log_return', 'entry_trade_value', 'exit_trade_value', 'signed_trade_qty', 'trade_gross_pnl')

### Add Transaction Fees

In [ ]:
trades = research.add_tx_fees(trades, maker_fee, taker_fee)
trades.select('datetime', 'entry_trade_value', 'exit_trade_value', 'tx_fee_maker', 'tx_fee_taker')

### Add Trade Net PnL

In [ ]:
trades = trades.with_columns(
    (pl.col('trade_gross_pnl') - pl.col('tx_fee_taker')).alias('trade_net_taker_pnl')
)
trades.select('datetime', 'open', 'close', 'close_log_return', 'y_hat', 'entry_trade_value', 'exit_trade_value', 'trade_gross_pnl', 'trade_net_taker_pnl')

### Display Equity Curve (Compounding)

In [ ]:
trades = research.add_equity_curve(trades, capital, 'trade_net_taker_pnl', 'taker')
research.plot_static_timeseries(trades, sym, 'equity_curve_taker', time_interval)

### Calculate Total Net Return for Compounding Trade Sizing

In [ ]:
compound_total_net_return = trades['equity_curve_taker'][-1] / capital - 1
compound_total_net_return

In [ ]:
constant_sizing_net_return

In [ ]:
np.round(compound_total_net_return - constant_sizing_net_return, 4)

## Key Strategy Decision #3: Leverage

In [ ]:
# leverage is borrowing money to increase our trade size and amplify our profits

In [ ]:
# For stocks, typical retail margin allows up to 2x overnight leverage (Reg T)
# Some prop firms offer higher leverage, but let's start with a conservative 2x
leverage = 2
leverage * capital

In [ ]:
# in theory, we are multiplying our trade size to increase our returns

In [ ]:
# we are NOT multiplying our trade returns

In [ ]:
# just remember, it amplifies BOTH your profit and losses so it's important to have a positive expected value

In [ ]:
# leverage only works when you have a high Sharpe model. The more risk you reduce, the more you reduce drawdowns and the more leverage you can safely use

In [ ]:
# key decision 3: should we use leverage? if so, how much? What's the sweet spot?

In [ ]:
# it's not a golden utopia for multiplying profits — it can easily wipe you out

In [ ]:
trades = research.add_compounding_trades(trades, capital, leverage, maker_fee, taker_fee)
trades

In [ ]:
trades['equity_curve_taker'][-1] / capital - 1

## What if we increase to 4x leverage?

In [ ]:
trades = research.add_compounding_trades(trades, capital, 4, maker_fee, taker_fee)

In [ ]:
trades['equity_curve_taker'][-1] / capital - 1

In [ ]:
# reason for this return is the combination of model's edge, compounding and leverage

In [ ]:
# this is possible on a small scale because we are trading such small size that we are not moving markets

In [ ]:
# show win rate to confirm there's no manipulation to make the PnL look good

### Win Rate hasn't changed

In [ ]:
trades.select((pl.col('trade_log_return') > 0).mean())

### Factor Liquidation (price moves against us too much)

In [ ]:
# Liquidation is when we go bust. If we use too much leverage, a small price change can wipe us out.

In [ ]:
# leverage is a double-edged sword — you can amplify profits, but too much leverage and you can wipe out all your money

In [ ]:
# For stock margin accounts:
# FINRA requires a 25% maintenance margin (equity must stay above 25% of position value)
# This is stricter than crypto exchanges (~0.5%) but stocks are regulated differently

In [ ]:
maintenance_margin = 0.25  # FINRA standard maintenance margin for US equities

def long_liquidation_price(p, l, mmr):
    return (p * l) / (l + 1 - mmr * l)

def short_liquidation_price(p, l, mmr):
    return (p * l) / (l - 1 + mmr * l)

### show how leverage affects long positions

In [ ]:
# entry price = $200, leverage = 1.5x: liquidated if price drops to...
long_liquidation_price(200, 1.5, maintenance_margin)

In [ ]:
# entry price = $200, leverage = 2x: liquidated if price drops to...
long_liquidation_price(200, 2, maintenance_margin)

In [ ]:
# entry price = $200, leverage = 3x: liquidated if price drops to...
long_liquidation_price(200, 3, maintenance_margin)

In [ ]:
# entry price = $200, leverage = 3.5x: notice how close to entry price this gets!
# With 25% maintenance margin, max viable leverage approaches 4x
long_liquidation_price(200, 3.5, maintenance_margin)

### show how leverage affects short positions

In [ ]:
# entry price = $200, leverage = 1.5x: liquidated if price rises to...
short_liquidation_price(200, 1.5, maintenance_margin)

In [ ]:
short_liquidation_price(200, 2, maintenance_margin)

In [ ]:
short_liquidation_price(200, 3, maintenance_margin)

In [ ]:
short_liquidation_price(200, 3.5, maintenance_margin)

### Add Liquidation Prices

In [ ]:
leverage

In [ ]:
trades = trades.with_columns(
    pl.when(pl.col('dir_signal') == 1)   # long position
      .then(
          (pl.col('open') * leverage)
          / (leverage + 1 - maintenance_margin * leverage)
      )
      .when(pl.col('dir_signal') == -1)  # short position
      .then(
          (pl.col('open') * leverage)
          / (leverage - 1 + maintenance_margin * leverage)
      )
      .otherwise(None)
      .alias('liquidation_price')
)
trades.select('datetime', 'open', 'high', 'low', 'close', 'liquidation_price', 'dir_signal')

### Add Liquidation Flag

In [ ]:
trades = trades.with_columns([
    # Worst price based on direction
    pl.when(pl.col('dir_signal') == 1)
      .then(pl.col('low'))
      .otherwise(pl.col('high'))
      .alias('worst_price'),

    # Liquidation flag
    pl.when(
        (pl.col('dir_signal') == 1) & (pl.col('low') <= pl.col('liquidation_price'))
    )
    .then(True)
    .when(
        (pl.col('dir_signal') == -1) & (pl.col('high') >= pl.col('liquidation_price'))
    )
    .then(True)
    .otherwise(False)
    .alias('liquidated')
])
trades.select('datetime', 'open', 'low', 'high', 'close', 'dir_signal', 'worst_price', 'liquidation_price', 'liquidated')

### Find Liquidated Trades

In [ ]:
trades.filter(pl.col('liquidated') == True)